# cALorie Notebook 46 — long-video gate balance (follow-up 1)

**Runtime: GPU (the Mac GPU with `CALORIE_DEVICE=mps`, or a Colab T4). Run all.**
Output: `calorie_long_video_gate_balance_report.zip`.

Runs `ml/data/long_video_train_v1/PLAN.md` follow-up 1 once (post-hoc, the only one allowed). It is
Notebook 45 with one change: long-video windows hold exactly 25% of each gate class. Notebook 45's
classifiers are reused frozen; only the three gates are retrained.

- Control: the frozen v2 seeds from Notebook 43 (parity-checked).
- Decision on the 5 dev videos with the evaluation-1 full-session scorer and the predeclared rule.
- Threshold 0.65 and fusion fixed; early stopping on V6-A validation only.
- The locked long-video test, the RepCount sealed test, and the creator benchmark are never read.


In [ ]:
%pip install -q scikit-learn pandas==2.2.3


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import random
import shutil
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

try:
    from IPython.display import display
except ImportError:  # local dry run outside IPython
    display = print

# Local dry-run switches. Colab leaves all of these unset.
DRY_RUN = os.environ.get("CALORIE_DRY_RUN") == "1"
LOCAL_REPO = os.environ.get("CALORIE_LOCAL_REPO")
WORK = Path(os.environ.get("CALORIE_WORK", "/content"))
WORK.mkdir(parents=True, exist_ok=True)

RUN_SEEDS = [42] if DRY_RUN else [42, 137, 2026]
CONTROL_SEEDS = [42, 137, 2026]
STAGE = "long_video_dev_followup1"  # dev decision only; no sealed or locked test is loaded here
LONG_LABEL_SHARE, LONG_DETECTED_SHARE = 0.75, 0.5  # PLAN.md section 3 details
DRY_RUN_LONG_WINDOWS_PER_VIDEO = 60
FIXED_FUSION = "mean_probability"
FIXED_THRESHOLD = 0.65
WINDOW = 64
STRIDE = 32
BATCH_SIZE = 64
MAX_EPOCHS = 1 if DRY_RUN else 40
PATIENCE = 8
GROUP_TEMPERATURE = 0.5  # frozen from V6-A.1
DRY_RUN_TRAIN_WINDOWS = 1500
DRY_RUN_VALIDATION_WINDOWS = 400
DRY_RUN_REP_TRAIN_CLIPS, DRY_RUN_REP_DEV_CLIPS = 40, 20
AUGMENTATION_CONFIG = {
    "horizontal_flip_probability": 0.5,
    "rotation_probability": 0.7,
    "max_rotation_degrees": 7.0,
    "speed_probability": 0.5,
    "speed_range": [0.85, 1.15],
    "frame_drop_probability": 0.35,
    "frame_drop_fraction_range": [0.05, 0.12],
    "coordinate_noise_probability": 0.8,
    "coordinate_noise_std": 0.008,
}
RETIRED_CLASSES = {"plank"}
NEW_TRAIN_GROUPS = {"youtube:Fk1wnNX4wQ8", "youtube:zxqCV_eQHZY"}
UNREVIEWED_GROUPS = {"youtube:oLsonHQJOUc", "youtube:c4TomZjzobY", "youtube:W2Bm9dfQjbk"}
STREAMS = {"joint": 34, "bone": 24, "velocity": 34}
SHARED_SOURCE_GROUPS = {
    "outdoor_crunch_shared_01": {"stu8_49", "stu9_56"},
    "orange_rig_squat_shared_01": {"stu1_69", "stu6_59"},
}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(RUN_SEEDS[0])
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# Local runs may opt into Apple-silicon GPUs with CALORIE_DEVICE=mps; Colab uses CUDA.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif os.environ.get("CALORIE_DEVICE") == "mps" and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Device:", DEVICE, "| dry run:", DRY_RUN)
if not DRY_RUN:
    assert DEVICE.type in ("cuda", "mps"), "Select a T4 runtime before training."


## Download pinned inputs


In [ ]:
TRAINING_COMMIT = "9874773e4681464bb00f5f0fb4dc3459caf75d6c"
DESIGN_COMMIT = "9544dfb4436148bed3328a2854b055f52b82893d"
ROLES_COMMIT = "24ed98d0dbc666510ee724acfa63186dad066710"
LONG_COMMIT = "93395df5df3b38eaf1c17680477c4ab8056e042d"
NB45_COMMIT = "5e77a260a4a48c00f399455052731e79895e8d6e"
INPUTS = {
    # name: (commit, repository path, sha256)
    "nb45_report.zip": (NB45_COMMIT, "ml/artifacts/long_video_training_v1/calorie_long_video_training_v1_report.zip", "81ec3493bae912a0048efb409edac80019b5d35d04bac44bcfde91d339cc72ca"),
    "long_roles.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/roles.csv", "8dd44526c99a6aff2d60ac3e6284e7965f2a0850a7249596173e01921e8b329d"),
    "score_long_video_test_v1.py": (LONG_COMMIT, "ml/scripts/score_long_video_test_v1.py", "df4b3db2be0ee8a8e0f861eecbbb53327210073ba61f21f648d97e04832bfedd"),
    "v2_stage_a.zip": (LONG_COMMIT, "ml/artifacts/repcount_training_v2_stage_a/calorie_repcount_training_v2_stage_a_report.zip", "83737851a0349e746a948bf1600095d494f588a1f3086e1548bd32b8fc131add"),
    "dev_diagnosis_metrics.csv": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/dev_diagnosis_v1/dev_metrics_by_model_video.csv", "19509d3e39caad5390746f08675692a41337bc417bf24e45291cfa1d7e0fd2a9"),
    "long_Ferc_0UlN4g.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_Ferc_0UlN4g.zip", "8852b90444fbab576e3bf94f703eede6e9f5a7390a6bc7cb73e6eb991451b4d2"),
    "long_R4S-eSPDXIg.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_R4S-eSPDXIg.zip", "5e7bc0b87e3b8f546daaeefb6c1b4ce5730fe0ecf7bb35ae1b9d70280dd502e2"),
    "long_5vmnb_T_YH4.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_5vmnb_T_YH4.zip", "418c530f1da0a25a481a21f34a3fcd2cc679fe6cd145514d5c607b08f6751e8b"),
    "long_uawtllYyD5g.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_uawtllYyD5g.zip", "24be57a06fb7fd939f0cd2ecbfcb92f27074c236f83141e68251d88bbb40eec7"),
    "long__cqwrzDHXsc.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1__cqwrzDHXsc.zip", "781ec0c26d2d8feb301e21593e6d182538b53a518e39f15afeee0b408c46a2e4"),
    "long_c9Y_0jdZaE0.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_c9Y_0jdZaE0.zip", "1200c038b57a95b69567dbe537d7ef3b26f8a2fb25264b6615b21d8c969bcf42"),
    "long_B1imXPO3oJE.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_B1imXPO3oJE.zip", "86b5ef2f2091c06ecf2a3a2a83c75e40802f0fb48f306d57f0ca88a818fb5a5c"),
    "long_jnJzNbtuR2Y.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_jnJzNbtuR2Y.zip", "3240a0623805923f7c017836fb9501e6fd722f365ee0c66d7cc1c8cfed73f9f4"),
    "long_06iVfPEDX3Y.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_06iVfPEDX3Y.zip", "a994f9c3edcdea3e9ed278048f4a794e0abf4de38ea838a3b8d0b6632be054a6"),
    "long_vvSRFH3jGEY.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_vvSRFH3jGEY.zip", "34e0e2158bfa5095ce1eea5051f60cb1b5da4d23e1e67a4745a2b5509987dfaf"),
    "long_yNN12hIAUyo.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_yNN12hIAUyo.zip", "481aac8b215db77a88afcadf3a2a65ab77f019884d1fa91c87034bed1f5c41d4"),
    "long_pKkgzbMJzy8.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_pKkgzbMJzy8.zip", "bbd47d32c22b3619f23c9182948d3148ea8abbe69448a1cc6d46f97556ad5eb9"),
    "long_Npm2PHZXs68.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_Npm2PHZXs68.zip", "e5c91153d0546eae2c55abf972333ffe124097a725ee06722b12b6cdedb9d122"),
    "long_weY7aAK9lEY.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_weY7aAK9lEY.zip", "b570b3b9632ae7bcaca0c4e43377d2d543708c48e91b2ae342e00c124f70c07b"),
    "long_5Cmw0wvYWP0.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_5Cmw0wvYWP0.zip", "8584e17e1286ca972e1b0958930909d3ba9cd58753e2161ca01a42b6ac243358"),
    "long_QrRDpD8xZfo.zip": (LONG_COMMIT, "ml/artifacts/long_video_train_v1/long_video_train_v1_QrRDpD8xZfo.zip", "a43e79c49964de07d6ad6afd8421e482988864ae0f53ace17f212a6f929bfcbc"),
    "long_ann_Ferc_0UlN4g.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/Ferc_0UlN4g.csv", "10d6a999ed8ec711ec0cb1dcdb9c240fb8028a55371abaa6c4e34484e6812aae"),
    "long_ann_R4S-eSPDXIg.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/R4S-eSPDXIg.csv", "acbb156fe760f06d411f5c5f8bf33ea39ebc7f6f8bb31d744849d0953767d636"),
    "long_ann_5vmnb_T_YH4.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/5vmnb_T_YH4.csv", "cfce87aba04f7e247063896b1461c08f1ca360b51c4e58b881456e3d1877abcb"),
    "long_ann_uawtllYyD5g.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/uawtllYyD5g.csv", "47d20e06f7e4efec3a0b93dd9ecdedfa90e4d640b3b1488ba5bfe627c38beffb"),
    "long_ann__cqwrzDHXsc.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/_cqwrzDHXsc.csv", "742201e2c9ad801311263eb62b62582419cd69b99629a86d305b0f631f70ed2a"),
    "long_ann_c9Y_0jdZaE0.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/c9Y_0jdZaE0.csv", "360eab7626a7bfc40c1325bf6f002f688add186712e9f2a9a82132dcaa0862ab"),
    "long_ann_B1imXPO3oJE.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/B1imXPO3oJE.csv", "2303bfd805b6d50f4dbe3a92cedd6da633e1ce01eca9ab72e17812abc22c7628"),
    "long_ann_jnJzNbtuR2Y.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/jnJzNbtuR2Y.csv", "b15b9cb48d191f26afa750309feb96e65e7edb8f8d6e1a5502803445205362b6"),
    "long_ann_06iVfPEDX3Y.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/06iVfPEDX3Y.csv", "d574038a63da2a230b11a22b2780d3ea0493712a1d13a24f64a31d11b4f2a655"),
    "long_ann_vvSRFH3jGEY.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/vvSRFH3jGEY.csv", "4a4a085172c37350323ea94f622a88b416f581b2903bdb8ef1bbf411c7d1a111"),
    "long_ann_yNN12hIAUyo.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/yNN12hIAUyo.csv", "987443215e320854c384f3a24d66015f7759dc5df45e03b81a70d2a3687d1027"),
    "long_ann_pKkgzbMJzy8.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/pKkgzbMJzy8.csv", "7d77591c108f0eb5f83c44cdde7bbb4886c9af54561dd3f4bec7b8d5e6df87e1"),
    "long_ann_Npm2PHZXs68.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/Npm2PHZXs68.csv", "24a548f2b3eea0d9ec3f0adc05eb25325eaefa1464de2420ab4f6e40329b927f"),
    "long_ann_weY7aAK9lEY.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/weY7aAK9lEY.csv", "73e721d5f75b0dcfec9574047d890a3e062db560018a32855a8b04acea1d8104"),
    "long_ann_5Cmw0wvYWP0.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/5Cmw0wvYWP0.csv", "299500a64a489e0f796f53420e0d90bd14ac2e5d228b927407069e3281cdba7b"),
    "long_ann_QrRDpD8xZfo.csv": (LONG_COMMIT, "ml/data/long_video_train_v1/annotations/QrRDpD8xZfo.csv", "ffefdab9aaa451fc30b7ef67850047e38b47b3ad9f1dc0a63a906b03aa25e1de"),
    "calorie_workoutfitness_landmarks_v1.zip": (TRAINING_COMMIT, "ml/artifacts/v1/landmarks/calorie_workoutfitness_landmarks_v1.zip", "6dd4828e73f1ac1cd83a175de5bc1caf8465cbf61b72777467fba540af339aa9"),
    "calorie_mmfit_landmarks_v1.zip": (TRAINING_COMMIT, "ml/artifacts/v1/landmarks/calorie_mmfit_landmarks_v1.zip", "2d3bff2c52842b276562e749fd7eaefe4925385efa7ff8c242c82ba1803413f1"),
    "calorie_mmfit_unknown_v1.zip": (TRAINING_COMMIT, "ml/artifacts/v1/landmarks/calorie_mmfit_unknown_v1.zip", "b0b113b3c5c7fc501b6b3ac4a7d2dade5fb4bccf3416d2eb85fa8b9197ae5596"),
    "calorie_youtube_mixed_landmarks_v3.zip": (TRAINING_COMMIT, "ml/artifacts/v2/landmarks/calorie_youtube_mixed_landmarks_v3.zip", "bd0426eda35654a95a8bac70b0357ba484e82bd29e6b3f71760fcc743f432cf4"),
    "calorie_youtube_additional_landmarks_v1.zip": (TRAINING_COMMIT, "ml/artifacts/v3/landmarks/calorie_youtube_additional_landmarks_v1.zip", "0c60478086400fdfa6974afc6b091023b46e10783a71e40929aaa2b646d02743"),
    "calorie_youtube_training_landmarks_v4.zip": (TRAINING_COMMIT, "ml/artifacts/v3/landmarks/calorie_youtube_training_landmarks_v4.zip", "e0588c9a4a7d0257f710fec74b657e863cda5e0bc3285a640443863c9ede2b19"),
    "calorie_workoutfitness_gate_hardneg_v1.zip": (TRAINING_COMMIT, "ml/artifacts/v5/landmarks/calorie_workoutfitness_gate_hardneg_v1.zip", "b3aab2dbcefd709f1977c4d8d3f7d0326b50444ab44db4ba6b0cf64bbae0f4f6"),
    "calorie_youtube_long_workouts_v2_reviewed_clips.zip": (TRAINING_COMMIT, "ml/artifacts/youtube_long_v2/reviewed_clips/calorie_youtube_long_workouts_v2_reviewed_clips.zip", "84fb55b981820af702a94f0c0070960b24fdd1acb617097f1838c782cd942284"),
    "hard_positive_windows.csv": (TRAINING_COMMIT, "ml/artifacts/v5/hard_positive_mining/hard_positive_windows.csv", "5a4b73c04ee85659570e7931ea0e84b35c0abeb16fd3f08238e629f6eca091c5"),
    "split_assignments.csv": (TRAINING_COMMIT, "ml/artifacts/v6a1/model/split_assignments.csv", "8c013695fe095ceccc4ff65722d2a26e5a309674f5b877b840b71ba73dc085d6"),
    "baseline_test_predictions.csv": (TRAINING_COMMIT, "ml/artifacts/v6c/model/test_window_predictions.csv", "b34b6cada90355682ef7aa225e3d18dad0960dd79ce32856ebb524356734326c"),
    "temporal_checkpoint.pt": (TRAINING_COMMIT, "ml/artifacts/v6e_complete_action_temporal/model/two_stage_tcn_v6e_temporal.pt", "c30ecd625d77f31380a503f881eff5c540b8a7cdbbb71a39276c33b908947b92"),
    "repcount_report.zip": (DESIGN_COMMIT, "ml/artifacts/repcount_external_development_v1/calorie_repcount_external_development_v1_report.zip", "fddd4e63f1162c1b48d1d81165c6e204127226824e5af500f452161f8a7219ef"),
    "training_v1_roles.csv": (ROLES_COMMIT, "ml/data/repcount_intake_v2/training_v1_roles.csv", "16f4008b9322c6c92c91192f050ea5d58a9b5dde231b1bcaa8a8b8f20ff6186e"),
    "intake_pilot.zip": (ROLES_COMMIT, "ml/artifacts/repcount_intake_v2/pilot/calorie_repcount_intake_v2_pilot.zip", "e9bb439a62ea224120aab256809a6d3dd6e737cb487d7a7a4b7cf8c5cbf54bae"),
    "intake_shard0.zip": (ROLES_COMMIT, "ml/artifacts/repcount_intake_v2/full/calorie_repcount_intake_v2_shard0.zip", "a967ee4e460b71b8b417fac5bf0995a5681ec904f4ac22c30f521ffb8dd44157"),
    "intake_shard1.zip": (ROLES_COMMIT, "ml/artifacts/repcount_intake_v2/full/calorie_repcount_intake_v2_shard1.zip", "17e55e624a923689dec923fc5f4b2ec98cbdcea0b83742351271aa5324c74805"),
    "intake_shard2.zip": (ROLES_COMMIT, "ml/artifacts/repcount_intake_v2/full/calorie_repcount_intake_v2_shard2.zip", "1bfe214e8fa300848bf36a0b0d3d128dbfb05e3b640f345ebabe4421e903529c"),
    "intake_shard3.zip": (ROLES_COMMIT, "ml/artifacts/repcount_intake_v2/full/calorie_repcount_intake_v2_shard3.zip", "ff0c1537ccc87d6bfd9c5538e092047e86a1d08a76cf4576148f022787fc6bdf"),
    "intake_shard4.zip": (ROLES_COMMIT, "ml/artifacts/repcount_intake_v2/full/calorie_repcount_intake_v2_shard4.zip", "b16aa78ffa706cdb898805930b2930db779963b531e7fa61ba152c0a4d4d0708"),
    "intake_shard5.zip": (ROLES_COMMIT, "ml/artifacts/repcount_intake_v2/full/calorie_repcount_intake_v2_shard5.zip", "7cda0e72856e6f49660c557322983d1cc9d549bedae6b236a63e39f1972559f5"),
    "intake_shard6.zip": (ROLES_COMMIT, "ml/artifacts/repcount_intake_v2/full/calorie_repcount_intake_v2_shard6.zip", "4cc4e1ed4f0d7e82fd611e9322fb87baaa36f285158efb7e1bc5637fa7ce4b36"),
    "intake_shard7.zip": (ROLES_COMMIT, "ml/artifacts/repcount_intake_v2/full/calorie_repcount_intake_v2_shard7.zip", "325be3d532e6aad92079a633500e5ccb8ed7f86a57fe44f5887f95b24fd2af9a"),
    "intake_makeup.zip": (ROLES_COMMIT, "ml/artifacts/repcount_intake_v2/full/calorie_repcount_intake_v2_makeup.zip", "94a73cbf1177c8a1428135e54b337952e8eb6eebef89d8c4a974cd82c57870c9"),
    "v6f_joint_gate.pt": (DESIGN_COMMIT, "ml/artifacts/v6f_multistream_gate/model/joint_gate.pt", "a01b0f9c00bcd69150fbd10db385fddb5e3c8b35497acae620dc1fa35f8f6cfb"),
    "v6f_bone_gate.pt": (DESIGN_COMMIT, "ml/artifacts/v6f_multistream_gate/model/bone_gate.pt", "ef7d3e19a7c237e99c7cad956b6f39e13f5aa8a36ff965c167ee690f4d400707"),
    "v6f_velocity_gate.pt": (DESIGN_COMMIT, "ml/artifacts/v6f_multistream_gate/model/velocity_gate.pt", "3af308a69145981ca0a5ea79b243b3d0b04d18f7c75ff49c59a088e9ce50788c"),
    "nb31_seed_metrics.csv": (DESIGN_COMMIT, "ml/artifacts/v6f_seed_stability/seed_metrics.csv", "a2fec12319d63ce6b65615acabb3a456b26dab46d88b9cc76644ee115c29d76e"),
    "control_seed_42_joint_gate.pt": (DESIGN_COMMIT, "ml/artifacts/v6f_seed_stability/seed_42_joint_gate.pt", "06953c97bf51b659be35d362637701bd5c17c78b5fd67bb8adf2c80ae22de210"),
    "control_seed_42_bone_gate.pt": (DESIGN_COMMIT, "ml/artifacts/v6f_seed_stability/seed_42_bone_gate.pt", "2d0f06da85f968dc024322db1ed006f2404545dbaf45aadc1cf629ebf55a2161"),
    "control_seed_42_velocity_gate.pt": (DESIGN_COMMIT, "ml/artifacts/v6f_seed_stability/seed_42_velocity_gate.pt", "bf0f5f1b6532e191003a1efdcad2b1303f3d9e41b57df7075c193ef0515069d6"),
    "control_seed_137_joint_gate.pt": (DESIGN_COMMIT, "ml/artifacts/v6f_seed_stability/seed_137_joint_gate.pt", "a7c8aed60f96476ee743868c1518628a7d7bb38afb141731476b4fd28a5e507c"),
    "control_seed_137_bone_gate.pt": (DESIGN_COMMIT, "ml/artifacts/v6f_seed_stability/seed_137_bone_gate.pt", "8bf896c9bf17a46f508f7cf4b079b6729e3b0fe387f3bc231fc39331e1e7069a"),
    "control_seed_137_velocity_gate.pt": (DESIGN_COMMIT, "ml/artifacts/v6f_seed_stability/seed_137_velocity_gate.pt", "7135cebc01c1d7c791f0b7288ff293274a5ded620dbaaab1ddd7bfe9b1c5c943"),
    "control_seed_2026_joint_gate.pt": (DESIGN_COMMIT, "ml/artifacts/v6f_seed_stability/seed_2026_joint_gate.pt", "32d3d29d62be3ceaf3412f6b163665d84e2edb4879b84b3c9e198f0a8f2f0aaa"),
    "control_seed_2026_bone_gate.pt": (DESIGN_COMMIT, "ml/artifacts/v6f_seed_stability/seed_2026_bone_gate.pt", "50b7b1ed5e347ad7021faa23fea38e09a7fb618587f53528e181f2ea7cff2948"),
    "control_seed_2026_velocity_gate.pt": (DESIGN_COMMIT, "ml/artifacts/v6f_seed_stability/seed_2026_velocity_gate.pt", "404b5ef7df394f6a99806021227ad4d85957b7809a33876e057063b0787de69a"),
}
assert not any("youtube_creator_benchmark" in path for _, path, _ in INPUTS.values()), "Locked benchmark must not be an input."
assert not any(path.startswith("ml/data/long_video_test_v1") or "artifacts/long_video_test_v1" in path for _, path, _ in INPUTS.values()), "Locked long-video test must not be an input."
LANDMARK_ARCHIVES = [name for name in INPUTS if name.startswith("calorie_") and name.endswith(".zip")]
INPUT_ROOT = WORK / "inputs"
INPUT_ROOT.mkdir(exist_ok=True)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch(name):
    commit, relative, expected = INPUTS[name]
    destination = INPUT_ROOT / name
    if destination.exists() and sha256(destination) == expected:
        print("Cached:", name)
        return destination
    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    if LOCAL_REPO:
        shutil.copy2(Path(LOCAL_REPO) / relative, temporary)
    else:
        url = f"https://raw.githubusercontent.com/bariscelikk1/cALorie/{commit}/{relative}"
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=300) as response, temporary.open("wb") as output:
            shutil.copyfileobj(response, output)
    actual = sha256(temporary)
    assert actual == expected, f"SHA-256 mismatch for {name}: {actual} (expected {expected}). Re-run the cell; if it repeats, the pinned file changed."
    temporary.replace(destination)
    print(f"{name}: {destination.stat().st_size / 1e6:.1f} MB")
    return destination


PATHS = {name: fetch(name) for name in INPUTS}
for name in LANDMARK_ARCHIVES:
    with zipfile.ZipFile(PATHS[name]) as archive:
        assert archive.testzip() is None, f"Corrupt ZIP: {name}"
hard_positive_frame = pd.read_csv(PATHS["hard_positive_windows.csv"])
assert hard_positive_frame.window_key.is_unique
baseline_predictions = pd.read_csv(PATHS["baseline_test_predictions.csv"])
nb31_seed_metrics = pd.read_csv(PATHS["nb31_seed_metrics.csv"])

SOURCE_ROOT = WORK / "training_sources"
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir()
for index, name in enumerate(LANDMARK_ARCHIVES):
    destination = SOURCE_ROOT / f"source_{index}"
    destination.mkdir()
    with zipfile.ZipFile(PATHS[name]) as archive:
        archive.extractall(destination)

frames = []
for manifest_path in SOURCE_ROOT.rglob("manifest.csv"):
    frame = pd.read_csv(manifest_path).fillna("")
    frame["npz_path"] = frame["output_path"].map(lambda value: str(manifest_path.parent / value))
    frames.append(frame)
assert frames, "No manifest.csv was found."
manifest = pd.concat(frames, ignore_index=True)
manifest = manifest[(manifest.status == "ok") & manifest.npz_path.map(lambda value: Path(value).exists())].copy()
if "group_id" not in manifest or manifest.group_id.eq("").any():
    manifest["group_id"] = manifest.apply(lambda row: f"{row['source_dataset']}:{row['clip_id']}", axis=1)
assert NEW_TRAIN_GROUPS <= set(manifest.group_id)
print("Valid clips:", len(manifest), "Groups:", manifest.group_id.nunique())


## Rebuild Notebook 31's exact training pool


In [ ]:
def resample_sequence(sequence, source_fps, target_fps=15):
    if not source_fps or abs(float(source_fps) - target_fps) < 0.1:
        return sequence
    duration = (len(sequence) - 1) / float(source_fps)
    target_length = max(2, int(round(duration * target_fps)) + 1)
    old_x = np.linspace(0, duration, len(sequence))
    new_x = np.linspace(0, duration, target_length)
    flat = sequence.reshape(len(sequence), -1)
    sampled = np.stack([np.interp(new_x, old_x, flat[:, index]) for index in range(flat.shape[1])], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


windows, original_labels, groups = [], [], []
clip_ids, sources, starts = [], [], []
for row in manifest.itertuples(index=False):
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    if sequence.ndim != 3 or sequence.shape[1:] != (17, 4):
        print("Skipping unexpected shape:", row.npz_path, sequence.shape)
        continue
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    clip_starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
    if len(sequence) > WINDOW and clip_starts[-1] != len(sequence) - WINDOW:
        clip_starts.append(len(sequence) - WINDOW)
    for start in clip_starts:
        window = sequence[start:start + WINDOW]
        if len(window) < WINDOW:
            window = np.concatenate([window, np.repeat(window[-1:], WINDOW - len(window), axis=0)])
        windows.append(window)
        original_labels.append(row.canonical_class)
        groups.append(row.group_id)
        clip_ids.append(row.clip_id)
        sources.append(row.source_dataset)
        starts.append(start)

X = np.asarray(windows, dtype=np.float32)
original_labels = np.asarray(original_labels, dtype=object)
groups = np.asarray(groups, dtype=object)
clip_ids = np.asarray(clip_ids, dtype=object)
sources = np.asarray(sources, dtype=object)
starts = np.asarray(starts, dtype=np.int64)
window_keys = np.asarray([f"{s}|{g}|{c}|{t}" for s, g, c, t in zip(sources, groups, clip_ids, starts)], dtype=object)

frozen_split = pd.read_csv(PATHS["split_assignments.csv"])
assert frozen_split.groupby("group_id")["split"].nunique().max() == 1
group_to_split = frozen_split[["group_id", "split"]].drop_duplicates("group_id").set_index("group_id")["split"].to_dict()
old_evidence_mask = np.asarray([group in group_to_split for group in groups])
assigned_split = np.asarray([group_to_split.get(group, "") for group in groups], dtype=object)
new_train_mask = np.isin(groups, list(NEW_TRAIN_GROUPS))
assigned_split[new_train_mask] = "train"
missing_groups = set(groups[assigned_split == ""])
assert missing_groups <= UNREVIEWED_GROUPS, "Unexpected unresolved groups: " + str(sorted(missing_groups))
labels = np.where(np.isin(original_labels, list(RETIRED_CLASSES)), "unknown", original_labels).astype(object)
ALL_CLASSES = sorted(np.unique(labels).tolist())
SUPPORTED_CLASSES = [name for name in ALL_CLASSES if name != "unknown"]
assert len(SUPPORTED_CLASSES) == 14, SUPPORTED_CLASSES
supported_to_id = {name: index for index, name in enumerate(SUPPORTED_CLASSES)}
assert set(window_keys[old_evidence_mask]) == set(frozen_split.window_key), "Reviewed window evidence changed since V6-A.1."


def estimate_period(sequence, minimum=12, maximum=90):
    xy = sequence[:, :, :2]
    energy = np.r_[0.0, np.linalg.norm(np.diff(xy, axis=0), axis=2).mean(axis=1)]
    energy = np.convolve(energy, np.ones(5) / 5.0, mode="same")
    centered = energy - energy.mean()
    upper = min(maximum, len(sequence) // 2)
    if upper <= minimum:
        return max(minimum, len(sequence))
    scores = []
    for lag in range(minimum, upper + 1):
        left, right = centered[:-lag], centered[lag:]
        denom = np.linalg.norm(left) * np.linalg.norm(right)
        scores.append(float(np.dot(left, right) / max(denom, 1e-8)))
    return minimum + int(np.argmax(scores))


def phase_resample(segment, target_length=WINDOW):
    xy = segment[:, :, :2]
    step_motion = np.r_[0.0, np.linalg.norm(np.diff(xy, axis=0), axis=2).mean(axis=1)]
    progress = np.cumsum(step_motion + 1e-5)
    progress = (progress - progress[0]) / max(progress[-1] - progress[0], 1e-8)
    targets = np.linspace(0.0, 1.0, target_length)
    flat = segment.reshape(len(segment), -1)
    sampled = np.stack([np.interp(targets, progress, flat[:, column]) for column in range(flat.shape[1])], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


temporal_windows, temporal_labels, temporal_groups = [], [], []
temporal_clip_ids, temporal_sources, temporal_starts, temporal_keys = [], [], [], []
for row in manifest.itertuples(index=False):
    canonical = "unknown" if row.canonical_class in RETIRED_CLASSES else row.canonical_class
    if (group_to_split.get(row.group_id) != "train" and row.group_id not in NEW_TRAIN_GROUPS) or canonical == "unknown":
        continue
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    if len(sequence) < 24:
        continue
    period = estimate_period(sequence)
    candidate_starts = list(range(0, max(1, len(sequence) - period + 1), period))
    if len(candidate_starts) > 6:
        candidate_starts = [candidate_starts[i] for i in np.linspace(0, len(candidate_starts) - 1, 6, dtype=int)]
    for cycle_index, start in enumerate(candidate_starts):
        end = min(len(sequence), start + period)
        if end - start < 12:
            continue
        temporal_windows.append(phase_resample(sequence[start:end]))
        temporal_labels.append(canonical)
        temporal_groups.append(row.group_id)
        temporal_clip_ids.append(row.clip_id)
        temporal_sources.append(row.source_dataset)
        temporal_starts.append(start)
        temporal_keys.append(f"phase64|{row.source_dataset}|{row.group_id}|{row.clip_id}|{cycle_index}")

assert temporal_windows, "No complete-action candidates were built."
X = np.concatenate([X, np.asarray(temporal_windows, dtype=np.float32)])
original_labels = np.concatenate([original_labels, np.asarray(temporal_labels, dtype=object)])
labels = np.concatenate([labels, np.asarray(temporal_labels, dtype=object)])
groups = np.concatenate([groups, np.asarray(temporal_groups, dtype=object)])
clip_ids = np.concatenate([clip_ids, np.asarray(temporal_clip_ids, dtype=object)])
sources = np.concatenate([sources, np.asarray(temporal_sources, dtype=object)])
starts = np.concatenate([starts, np.asarray(temporal_starts, dtype=np.int64)])
window_keys = np.concatenate([window_keys, np.asarray(temporal_keys, dtype=object)])
assigned_split = np.concatenate([assigned_split, np.repeat("train", len(temporal_windows)).astype(object)])

train_idx = np.flatnonzero(assigned_split == "train")
val_idx = np.flatnonzero(assigned_split == "validation")
test_idx = np.flatnonzero(assigned_split == "test")
assert not (set(groups[train_idx]) & set(groups[val_idx]))
assert not (set(groups[train_idx]) & set(groups[test_idx]))
assert not (set(groups[val_idx]) & set(groups[test_idx]))
assert not (set(groups[train_idx]) & UNREVIEWED_GROUPS)
assert len(test_idx) == 1971, len(test_idx)
assert set(window_keys[test_idx]) == set(baseline_predictions.window_key)
baseline_truth_by_key = dict(zip(baseline_predictions.window_key, baseline_predictions.true_class))
assert all(baseline_truth_by_key[key] == truth for key, truth in zip(window_keys[test_idx], labels[test_idx])), "Frozen test labels changed."
print("Base windows:", X.shape, "| train", len(train_idx), "| validation", len(val_idx), "| test", len(test_idx))


## Add RepCount train and dev clips (sealed test excluded)


In [ ]:
ROLES = pd.read_csv(PATHS["training_v1_roles.csv"])
assert ROLES.video_id.is_unique and set(ROLES.role) == {"train", "dev", "sealed_test"}
assert (ROLES.groupby("role_group").role.nunique() == 1).all(), "A role group spans roles."
assert set(ROLES.label) <= set(ALL_CLASSES), set(ROLES.label) - set(ALL_CLASSES)
active = ROLES[ROLES.role != "sealed_test"].copy()  # stage A never reads sealed-test pose
if DRY_RUN:
    active = pd.concat([active[active.role == "train"].sample(DRY_RUN_REP_TRAIN_CLIPS, random_state=0),
                        active[active.role == "dev"].sample(DRY_RUN_REP_DEV_CLIPS, random_state=0)])
import io
need = set(active.video_id)


def zip_pose_index(name, prefix="report/"):
    """Map video_id -> (zip path, member, expected sha) using the archive's own checksum list."""
    with zipfile.ZipFile(PATHS[name]) as archive:
        listed = dict(line.split("  ", 1)[::-1] for line in archive.read(prefix + "SHA256SUMS.txt").decode().splitlines())
        index = {}
        for member in archive.namelist():
            if member.startswith(prefix + "pose/") and member.endswith(".npz"):
                relative = member[len(prefix):]
                index[Path(member).stem] = (PATHS[name], member, listed[relative])
    return index


pose_source = {}
for name in [n for n in INPUTS if n.startswith("intake_")]:  # make-up is last and wins
    pose_source.update(zip_pose_index(name))
v1_source = zip_pose_index("repcount_report.zip")
pose_source.update({k: v for k, v in v1_source.items() if k in need})
missing = need - set(pose_source)
assert not missing, f"No pose for {sorted(missing)[:5]}"


def load_pose(entry):
    path, member, expected = entry
    with zipfile.ZipFile(path) as archive:
        payload = archive.read(member)
    assert hashlib.sha256(payload).hexdigest() == expected, f"Checksum failed: {member}"
    with np.load(io.BytesIO(payload), allow_pickle=False) as item:
        assert abs(float(item["target_fps"]) - 15.0) < 1e-3
        pose = item["landmarks"].astype(np.float32)
    assert pose.ndim == 3 and pose.shape[1:] == (17, 4) and len(pose) > 0 and np.isfinite(pose).all()
    return pose


def resample_window(pose):
    old = np.arange(len(pose), dtype=np.float32)
    new = np.linspace(0, len(pose) - 1, WINDOW, dtype=np.float32)
    flat = pose.reshape(len(pose), -1)
    return np.stack([np.interp(new, old, flat[:, j]) for j in range(flat.shape[1])], 1).reshape(WINDOW, 17, 4).astype(np.float32)


def eval_windows(pose):
    """Notebook 37 rule (used for clip-level scoring)."""
    if len(pose) < WINDOW:
        return [0], [resample_window(pose)]
    starts_ = list(range(0, len(pose) - WINDOW + 1, STRIDE))
    if starts_[-1] != len(pose) - WINDOW:
        starts_.append(len(pose) - WINDOW)
    return starts_, [pose[s:s + WINDOW] for s in starts_]


def train_windows(pose, label):
    """Notebook 31 rule: fixed windows (padded when short) plus phase cycles for supported clips."""
    starts_ = list(range(0, max(1, len(pose) - WINDOW + 1), STRIDE))
    if len(pose) > WINDOW and starts_[-1] != len(pose) - WINDOW:
        starts_.append(len(pose) - WINDOW)
    out = []
    for s in starts_:
        w = pose[s:s + WINDOW]
        if len(w) < WINDOW:
            w = np.concatenate([w, np.repeat(w[-1:], WINDOW - len(w), axis=0)])
        out.append((f"fixed|{s}", s, w))
    if label != "unknown" and len(pose) >= 24:
        period = estimate_period(pose)
        cands = list(range(0, max(1, len(pose) - period + 1), period))
        if len(cands) > 6:
            cands = [cands[i] for i in np.linspace(0, len(cands) - 1, 6, dtype=int)]
        for k, s in enumerate(cands):
            e = min(len(pose), s + period)
            if e - s >= 12:
                out.append((f"phase64|{k}", s, phase_resample(pose[s:e])))
    return out


new_windows, rows = [], []
for r in active.itertuples(index=False):
    pose = load_pose(pose_source[r.video_id])
    if r.role == "train":
        for tag, s, w in train_windows(pose, r.label):
            new_windows.append(w); rows.append((r.video_id, r.label, r.role_group, "repcount_train", s, f"repcount|train|{r.video_id}|{tag}"))
    else:
        for s, w in zip(*eval_windows(pose)):
            new_windows.append(w); rows.append((r.video_id, r.label, r.role_group, "repcount_dev", s, f"repcount|dev|{r.video_id}|{s}"))
# V1 parity windows: the 30 Notebook 37 clips, scored only to reproduce frozen V6-F (never trained here).
with zipfile.ZipFile(PATHS["repcount_report.zip"]) as archive:
    v1_ids = sorted(pd.read_csv(archive.open("report/video_diagnostics.csv")).video_id)
for vid in v1_ids:
    for s, w in zip(*eval_windows(load_pose(v1_source[vid]))):
        new_windows.append(w); rows.append((vid, "v1", "v1", "v1_parity", s, f"repcount|v1parity|{vid}|{s}"))

meta = pd.DataFrame(rows, columns=["video_id", "label", "role_group", "split", "start", "window_key"])
rep_start = len(X)
X = np.concatenate([X, np.asarray(new_windows, dtype=np.float32)])
meta_labels = meta.label.where(meta.split != "v1_parity", "unknown").to_numpy(dtype=object)  # v1 parity rows are never trained
original_labels = np.concatenate([original_labels, meta_labels])
labels = np.concatenate([labels, meta_labels])
groups = np.concatenate([groups, ("repcount:" + meta.role_group).to_numpy(dtype=object)])
clip_ids = np.concatenate([clip_ids, meta.video_id.to_numpy(dtype=object)])
sources = np.concatenate([sources, np.repeat("repcount", len(meta)).astype(object)])
starts = np.concatenate([starts, meta.start.to_numpy(dtype=np.int64)])
window_keys = np.concatenate([window_keys, meta.window_key.to_numpy(dtype=object)])
assigned_split = np.concatenate([assigned_split, meta.split.to_numpy(dtype=object)])
y_gate = (labels != "unknown").astype(np.int64)
y_exercise = np.asarray([supported_to_id.get(name, -1) for name in labels], dtype=np.int64)
rep_train_idx = np.flatnonzero(assigned_split == "repcount_train")
dev_idx = np.flatnonzero(assigned_split == "repcount_dev")
v1p_idx = np.flatnonzero(assigned_split == "v1_parity")
assert window_keys.size == len(set(window_keys)), "Window keys must be unique."
assert len(test_idx) == 1971 and np.array_equal(test_idx, np.flatnonzero(assigned_split == "test"))
assert not set(groups[rep_train_idx]) & set(groups[dev_idx]), "Train and dev share a role group."
assert not set(groups[np.r_[rep_train_idx, dev_idx]]) & set(groups[np.r_[train_idx, val_idx, test_idx]])
assert not set(clip_ids[np.r_[rep_train_idx, dev_idx]]) & set(ROLES.loc[ROLES.role == "sealed_test", "video_id"])
dev_truth = dict(zip(active.video_id, active.label))
print("RepCount train windows:", len(rep_train_idx), "| dev windows:", len(dev_idx), "| v1 parity windows:", len(v1p_idx))
display(meta[meta.split != "v1_parity"].groupby(["split", "label"]).video_id.nunique().unstack(0, fill_value=0))


## Add the 11 long-video train videos (continuous windows)


In [ ]:
import importlib.util
import io
spec = importlib.util.spec_from_file_location("lvscore", PATHS["score_long_video_test_v1.py"])
lvscore = importlib.util.module_from_spec(spec); spec.loader.exec_module(lvscore)
assert lvscore.THRESHOLD == FIXED_THRESHOLD and lvscore.SUPPORTED == SUPPORTED_CLASSES

LONG_ROLES = pd.read_csv(PATHS["long_roles.csv"])
LONG_TRAIN = sorted(LONG_ROLES.loc[LONG_ROLES.role == "train", "video_id"])
LONG_DEV = sorted(LONG_ROLES.loc[LONG_ROLES.role == "dev", "video_id"])
assert len(LONG_TRAIN) == 11 and len(LONG_DEV) == 5 and not set(LONG_TRAIN) & set(LONG_DEV)
assert not set(LONG_ROLES.channel[LONG_ROLES.role == "train"]) & set(LONG_ROLES.channel[LONG_ROLES.role == "dev"]), "A channel spans roles."


def long_pose(vid):
    with zipfile.ZipFile(PATHS[f"long_{vid}.zip"]) as archive:
        listed = dict(line.split("  ", 1)[::-1] for line in archive.read(f"{vid}/SHA256SUMS.txt").decode().splitlines())
        payload = archive.read(f"{vid}/pose.npz")
    assert hashlib.sha256(payload).hexdigest() == listed["pose.npz"], f"Checksum failed: {vid}"
    with np.load(io.BytesIO(payload), allow_pickle=False) as item:
        assert abs(float(item["target_fps"]) - 15.0) < 1e-3
        return item["landmarks"].astype(np.float32), item["detected"].astype(bool)


def long_annotation(vid):
    ann = pd.read_csv(PATHS[f"long_ann_{vid}.csv"]).sort_values("start_s").reset_index(drop=True)
    assert set(ann.review_status) <= {"reviewed", "adjudicated"} and not ann.needs_second_review.astype(str).str.lower().eq("true").any()
    return ann


long_windows, long_rows, long_audit = [], [], []
for vid in LONG_TRAIN:
    pose, detected = long_pose(vid)
    ann = long_annotation(vid)
    frame_label, _, _ = lvscore.truth_at(np.arange(len(pose)) / 15.0, ann)
    kept = dropped_mixed = dropped_pose = 0
    starts_ = list(range(0, len(pose) - WINDOW + 1, STRIDE))
    if DRY_RUN:
        starts_ = starts_[::max(1, len(starts_) // DRY_RUN_LONG_WINDOWS_PER_VIDEO)]
    for s in starts_:
        values, counts = np.unique(frame_label[s:s + WINDOW], return_counts=True)
        if counts.max() < LONG_LABEL_SHARE * WINDOW:
            dropped_mixed += 1; continue
        if detected[s:s + WINDOW].mean() < LONG_DETECTED_SHARE:
            dropped_pose += 1; continue
        long_windows.append(pose[s:s + WINDOW]); long_rows.append((vid, values[counts.argmax()], s)); kept += 1
    long_audit.append({"video_id": vid, "windows_kept": kept, "dropped_mixed_label": dropped_mixed, "dropped_low_pose": dropped_pose})
long_meta = pd.DataFrame(long_rows, columns=["video_id", "label", "start"])
assert set(long_meta.label) <= set(ALL_CLASSES)
X = np.concatenate([X, np.asarray(long_windows, dtype=np.float32)])
original_labels = np.concatenate([original_labels, long_meta.label.to_numpy(dtype=object)])
labels = np.concatenate([labels, long_meta.label.to_numpy(dtype=object)])
groups = np.concatenate([groups, ("longvideo:" + long_meta.video_id).to_numpy(dtype=object)])
clip_ids = np.concatenate([clip_ids, long_meta.video_id.to_numpy(dtype=object)])
sources = np.concatenate([sources, np.repeat("long_video", len(long_meta)).astype(object)])
starts = np.concatenate([starts, long_meta.start.to_numpy(dtype=np.int64)])
window_keys = np.concatenate([window_keys, ("longvideo|" + long_meta.video_id + "|" + long_meta.start.astype(str)).to_numpy(dtype=object)])
assigned_split = np.concatenate([assigned_split, np.repeat("long_train", len(long_meta)).astype(object)])
y_gate = (labels != "unknown").astype(np.int64)
y_exercise = np.asarray([supported_to_id.get(name, -1) for name in labels], dtype=np.int64)
long_train_idx = np.flatnonzero(assigned_split == "long_train")
assert window_keys.size == len(set(window_keys))
assert not set(clip_ids[long_train_idx]) & set(LONG_DEV)
assert not set(groups[long_train_idx]) & set(groups[np.r_[train_idx, val_idx, test_idx, rep_train_idx, dev_idx]])
long_audit = pd.DataFrame(long_audit)
display(long_audit)
display(long_meta.groupby("label").size().rename("windows").to_frame())


## Sampler, augmentation, and datasets


In [ ]:
hard_weight_by_key = dict(zip(hard_positive_frame.window_key, hard_positive_frame.recommended_sample_weight))
hard_positive_weights = np.asarray([float(hard_weight_by_key.get(key, 1.0)) for key in window_keys], dtype=np.float64)
hard_positive_weights[labels == "unknown"] = 1.0
assert not (set(window_keys[val_idx]) & set(hard_weight_by_key))
assert not (set(window_keys[test_idx]) & set(hard_weight_by_key))
assert not (set(window_keys[np.r_[rep_train_idx, dev_idx]]) & set(hard_weight_by_key))
gate_hierarchy = np.where(y_gate == 1, "supported", "unknown")


def hierarchical_window_weights(indices, hierarchy_labels, extra_weights=None):
    table = pd.DataFrame({"index": indices, "hierarchy_label": hierarchy_labels[indices], "group": groups[indices]})
    class_count = table["hierarchy_label"].nunique()
    group_sizes = table.groupby(["hierarchy_label", "group"])["index"].transform("count").astype(float)
    table["group_mass"] = group_sizes.pow(GROUP_TEMPERATURE)
    class_group_mass = table.drop_duplicates(["hierarchy_label", "group"]).groupby("hierarchy_label")["group_mass"].sum()
    table["extra_weight"] = 1.0 if extra_weights is None else extra_weights[table["index"].to_numpy()]
    table["pair_extra_total"] = table.groupby(["hierarchy_label", "group"])["extra_weight"].transform("sum")
    weights = (
        1.0 / class_count
        * table["group_mass"].to_numpy() / class_group_mass.loc[table["hierarchy_label"]].to_numpy()
        * table["extra_weight"].to_numpy() / table["pair_extra_total"].to_numpy()
    )
    return weights / weights.mean()


def dry_run_subset(indices, size, seed):
    if not DRY_RUN or len(indices) <= size:
        return indices
    return np.sort(np.random.default_rng(seed).choice(indices, size, replace=False))


LEFT_RIGHT_PAIRS = [(1, 2), (3, 4), (5, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16)]


def augment_pose_window(sample):
    sample = sample.copy()
    config = AUGMENTATION_CONFIG
    if np.random.random() < config["horizontal_flip_probability"]:
        sample[:, :, 0] *= -1.0
        for left, right in LEFT_RIGHT_PAIRS:
            sample[:, [left, right], :] = sample[:, [right, left], :]
    if np.random.random() < config["rotation_probability"]:
        radians = np.deg2rad(np.random.uniform(-config["max_rotation_degrees"], config["max_rotation_degrees"]))
        cosine, sine = np.cos(radians), np.sin(radians)
        x = sample[:, :, 0].copy()
        y = sample[:, :, 1].copy()
        sample[:, :, 0] = cosine * x - sine * y
        sample[:, :, 1] = sine * x + cosine * y
    if np.random.random() < config["speed_probability"]:
        speed = np.random.uniform(*config["speed_range"])
        center = (WINDOW - 1) / 2.0
        positions = (np.arange(WINDOW) - center) * speed + center
        flat = sample.reshape(WINDOW, -1)
        flat = np.stack([np.interp(positions, np.arange(WINDOW), flat[:, column]) for column in range(flat.shape[1])], axis=1)
        sample = flat.reshape(WINDOW, 17, 4).astype(np.float32)
    if np.random.random() < config["frame_drop_probability"]:
        fraction = np.random.uniform(*config["frame_drop_fraction_range"])
        count = max(1, int(round(WINDOW * fraction)))
        dropped = np.random.choice(np.arange(1, WINDOW), size=count, replace=False)
        sample[dropped] = sample[dropped - 1]
    if np.random.random() < config["coordinate_noise_probability"]:
        sample[:, :, :2] += np.random.normal(0, config["coordinate_noise_std"], sample[:, :, :2].shape).astype(np.float32)
    return sample.astype(np.float32)


BONE_EDGES = [(5, 6), (5, 7), (7, 9), (6, 8), (8, 10), (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)]
ANGLE_TRIPLETS = [(5, 7, 9), (6, 8, 10), (7, 5, 11), (8, 6, 12), (5, 11, 13), (6, 12, 14), (11, 13, 15), (12, 14, 16)]
FEATURE_DIM = 100


def pose_to_features(sample):
    xy = sample[:, :, :2].astype(np.float32)
    velocity = np.zeros_like(xy)
    velocity[1:] = xy[1:] - xy[:-1]
    bones = np.stack([xy[:, child] - xy[:, parent] for parent, child in BONE_EDGES], axis=1)
    angles = []
    for first, center, last in ANGLE_TRIPLETS:
        first_vector = xy[:, first] - xy[:, center]
        last_vector = xy[:, last] - xy[:, center]
        denominator = np.linalg.norm(first_vector, axis=1) * np.linalg.norm(last_vector, axis=1)
        cosine = np.sum(first_vector * last_vector, axis=1) / np.maximum(denominator, 1e-6)
        angles.append(np.arccos(np.clip(cosine, -1.0, 1.0)) / np.pi)
    features = np.concatenate([xy.reshape(WINDOW, -1), velocity.reshape(WINDOW, -1), bones.reshape(WINDOW, -1), np.stack(angles, axis=1)], axis=1).astype(np.float32)
    assert features.shape == (WINDOW, FEATURE_DIM) and np.isfinite(features).all()
    return features


def pose_to_stream(sample, stream):
    xy = sample[:, :, :2].astype(np.float32)
    if stream == "joint":
        return xy.reshape(WINDOW, -1)
    if stream == "velocity":
        velocity = np.zeros_like(xy)
        velocity[1:] = xy[1:] - xy[:-1]
        return velocity.reshape(WINDOW, -1)
    if stream == "bone":
        return np.stack([xy[:, child] - xy[:, parent] for parent, child in BONE_EDGES], axis=1).reshape(WINDOW, -1)
    raise ValueError(stream)


class StreamDataset(Dataset):
    def __init__(self, indices, training, stream):
        self.indices, self.training, self.stream = np.asarray(indices), training, stream

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        actual = self.indices[item]
        sample = X[actual].copy()
        if self.training:
            sample = augment_pose_window(sample)
        return torch.from_numpy(pose_to_stream(sample, self.stream).T), torch.tensor(y_gate[actual], dtype=torch.long)


def make_stream_loaders(stream, seed, train_indices, train_weights, validation_indices):
    generator = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(torch.as_tensor(train_weights, dtype=torch.double), len(train_weights), replacement=True, generator=generator)
    return (
        DataLoader(StreamDataset(train_indices, True, stream), batch_size=BATCH_SIZE, sampler=sampler),
        DataLoader(StreamDataset(validation_indices, False, stream), batch_size=BATCH_SIZE, shuffle=False),
    )



class ExerciseDataset(Dataset):
    def __init__(self, indices, training):
        self.indices, self.training = np.asarray(indices), training

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        actual = self.indices[item]
        sample = X[actual].copy()
        if self.training:
            sample = augment_pose_window(sample)
        return torch.from_numpy(pose_to_features(sample).T), torch.tensor(y_exercise[actual], dtype=torch.long)


def make_exercise_loaders(seed, train_indices, train_weights, validation_indices):
    generator = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(torch.as_tensor(train_weights, dtype=torch.double), len(train_weights), replacement=True, generator=generator)
    return (
        DataLoader(ExerciseDataset(train_indices, True), batch_size=BATCH_SIZE, sampler=sampler),
        DataLoader(ExerciseDataset(validation_indices, False), batch_size=BATCH_SIZE, shuffle=False),
    )


gate_train = np.concatenate([dry_run_subset(train_idx, DRY_RUN_TRAIN_WINDOWS, 0), rep_train_idx, long_train_idx])  # the one change
gate_weights = hierarchical_window_weights(gate_train, gate_hierarchy, hard_positive_weights)
exercise_train = gate_train[y_exercise[gate_train] >= 0]
exercise_weights = hierarchical_window_weights(exercise_train, labels)

REPCOUNT_CAP = 0.5  # TRAINING_PLAN_V2.md: the only change from Notebook 42


def cap_repcount(indices, weights, hierarchy):
    """Scale RepCount and original windows so RepCount holds at most REPCOUNT_CAP of each class's mass."""
    w = weights.copy()
    rep = sources[indices] == "repcount"
    for target in set(hierarchy[indices]):
        m = hierarchy[indices] == target
        rep_mass, old_mass = w[m & rep].sum(), w[m & ~rep].sum()
        if rep_mass > 0 and old_mass > 0 and rep_mass / (rep_mass + old_mass) > REPCOUNT_CAP:
            total = rep_mass + old_mass  # class mass is preserved
            w[m & rep] *= REPCOUNT_CAP * total / rep_mass
            w[m & ~rep] *= (1 - REPCOUNT_CAP) * total / old_mass
    return w / w.mean()


gate_weights = cap_repcount(gate_train, gate_weights, gate_hierarchy)
exercise_weights = cap_repcount(exercise_train, exercise_weights, labels)

LONG_GATE_SHARE = 0.25  # PLAN.md follow-up 1: the only change from Notebook 45


def balance_long_video(indices, weights, hierarchy):
    """Rescale so long-video windows hold exactly LONG_GATE_SHARE of each class's mass (class mass preserved)."""
    w = weights.copy()
    longv = sources[indices] == "long_video"
    for target in set(hierarchy[indices]):
        m = hierarchy[indices] == target
        long_mass, other_mass = w[m & longv].sum(), w[m & ~longv].sum()
        assert long_mass > 0 and other_mass > 0, target
        total = long_mass + other_mass
        w[m & longv] *= LONG_GATE_SHARE * total / long_mass
        w[m & ~longv] *= (1 - LONG_GATE_SHARE) * total / other_mass
    return w / w.mean()


gate_weights = balance_long_video(gate_train, gate_weights, gate_hierarchy)
validation_indices = dry_run_subset(val_idx, DRY_RUN_VALIDATION_WINDOWS, 99)
exercise_validation = validation_indices[y_exercise[validation_indices] >= 0]

audit_rows = []
for name, idx, w, hierarchy in [("gate", gate_train, gate_weights, gate_hierarchy), ("exercise", exercise_train, exercise_weights, labels)]:
    rep, longv = sources[idx] == "repcount", sources[idx] == "long_video"
    for target in sorted(set(hierarchy[idx])):
        m = hierarchy[idx] == target
        audit_rows.append({"model": name, "target": target, "windows": int(m.sum()), "repcount_windows": int((m & rep).sum()),
                           "sampler_mass": float(w[m].sum() / w.sum()), "repcount_sampler_mass": float(w[m & rep].sum() / w.sum()),
                           "long_video_windows": int((m & longv).sum()), "long_video_sampler_mass": float(w[m & longv].sum() / w.sum())})
sampler_audit = pd.DataFrame(audit_rows)
assert (sampler_audit.repcount_sampler_mass <= REPCOUNT_CAP * sampler_audit.sampler_mass + 1e-9).all(), "Cap not applied."
gate_audit = sampler_audit[sampler_audit.model == "gate"]
assert np.allclose(gate_audit.long_video_sampler_mass, LONG_GATE_SHARE * gate_audit.sampler_mass), "Gate balance not applied."
display(sampler_audit)


## Models and trainer (Notebooks 26 and 31)


In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        blocks, current = [], input_features
        for level, output in enumerate([64, 96, 128, 128]):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=-1))


checkpoint = torch.load(PATHS["temporal_checkpoint.pt"], map_location=DEVICE, weights_only=False)
frozen_exercise_model = SmallTCN(FEATURE_DIM, len(SUPPORTED_CLASSES)).to(DEVICE)
frozen_exercise_model.load_state_dict(checkpoint["exercise_state_dict"])
frozen_exercise_model.eval()
for parameter in frozen_exercise_model.parameters():
    parameter.requires_grad = False


def load_gate(path, stream):
    gate_checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    assert gate_checkpoint["stream"] == stream and int(gate_checkpoint["input_features"]) == STREAMS[stream]
    model = SmallTCN(STREAMS[stream], 2).to(DEVICE)
    model.load_state_dict(gate_checkpoint["state_dict"])
    return model.eval()


def train_network(input_features, num_classes, loaders, seed, tag, stream):
    seed_everything(seed)
    model = SmallTCN(input_features, num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)
    best_path = WORK / f"best_{tag}_{seed}_{stream}.pt"
    best_f1, stale, history = -1.0, 0, []

    def epoch(loader, training):
        model.train(training)
        losses, truth, prediction = [], [], []
        with (torch.enable_grad() if training else torch.no_grad()):
            for features, targets in loader:
                features, targets = features.to(DEVICE), targets.to(DEVICE)
                if training:
                    optimizer.zero_grad(set_to_none=True)
                logits = model(features)
                loss = criterion(logits, targets)
                if training:
                    loss.backward()
                    optimizer.step()
                losses.append(loss.item() * len(targets))
                truth.extend(targets.cpu().numpy())
                prediction.extend(logits.argmax(1).cpu().numpy())
        return sum(losses) / len(truth), f1_score(truth, prediction, average="macro", zero_division=0)

    for epoch_number in range(1, MAX_EPOCHS + 1):
        train_loss, train_f1 = epoch(loaders[0], True)
        val_loss, val_f1 = epoch(loaders[1], False)
        scheduler.step(val_f1)
        history.append({"tag": tag, "seed": seed, "stream": stream, "epoch": epoch_number, "train_loss": train_loss,
                        "train_macro_f1": train_f1, "val_loss": val_loss, "val_macro_f1": val_f1})
        print(tag, seed, stream, epoch_number, round(train_f1, 3), round(val_f1, 3))
        if val_f1 > best_f1 + 1e-4:
            best_f1, stale = val_f1, 0
            torch.save(model.state_dict(), best_path)
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    return model.eval(), history, best_f1



def train_gate(stream, loaders, seed, tag):
    return train_network(STREAMS[stream], 2, loaders, seed, tag, stream)


def train_classifier(loaders, seed, tag):
    return train_network(FEATURE_DIM, len(SUPPORTED_CLASSES), loaders, seed, tag, "exercise")


## Fixed scoring


In [ ]:
TEST_CLASSES = ["unknown", "squat", "jumping_jack", "bench_press", "pull_up", "sit_up", "push_up"]
UNCOVERED = ["bicep_curl", "deadlift", "dumbbell_row", "lat_pulldown", "lateral_raise", "lunge", "shoulder_press", "tricep_extension"]


def infer_stream(model, indices, stream, batch_size=256):
    out = []
    with torch.no_grad():
        for s in range(0, len(indices), batch_size):
            batch = np.stack([pose_to_stream(X[i], stream) for i in indices[s:s + batch_size]]).transpose(0, 2, 1)
            out.append(model(torch.from_numpy(batch).to(DEVICE)).softmax(1).cpu().numpy())
    return np.concatenate(out)[:, 1]


def fused_score(gates, indices):
    return np.stack([infer_stream(gates[s], indices, s) for s in STREAMS], axis=1).mean(axis=1)


def class_probabilities(model, indices, batch_size=256):
    out = []
    with torch.no_grad():
        for s in range(0, len(indices), batch_size):
            batch = np.stack([pose_to_features(X[i]) for i in indices[s:s + batch_size]]).transpose(0, 2, 1)
            out.append(model(torch.from_numpy(batch).to(DEVICE)).softmax(1).cpu().numpy())
    return np.concatenate(out)


def internal_metrics(gate_scores, class_prob):
    pred = np.asarray([SUPPORTED_CLASSES[c] if s >= FIXED_THRESHOLD else "unknown" for s, c in zip(gate_scores, class_prob.argmax(1))])
    truth = labels[test_idx].astype(str)
    recall = {c: float((pred[truth == c] == c).mean()) for c in ALL_CLASSES if (truth == c).any()}
    return {"end_to_end_macro_f1": float(f1_score(truth, pred, labels=ALL_CLASSES, average="macro", zero_division=0)),
            "end_to_end_accuracy": float(accuracy_score(truth, pred)),
            "unknown_recall": recall["unknown"],
            "uncovered_mean_recall": float(np.mean([recall[c] for c in UNCOVERED if c in recall])),
            **{f"recall__{c}": v for c, v in recall.items()}}


def error_type(target, prediction):
    if prediction == target:
        return "correct"
    if target != "unknown" and prediction == "unknown":
        return "gate_false_unknown"
    if target == "unknown":
        return "gate_false_accept"
    return "wrong_supported_class"


def clip_table(indices, gate_scores, class_prob, truth):
    """Notebook 37 rule: mean window gate score vs 0.65; mean window class probability."""
    video = clip_ids[indices]
    rows = []
    for vid in pd.unique(video):
        m = video == vid
        oracle = SUPPORTED_CLASSES[int(class_prob[m].mean(0).argmax())]
        gate = float(gate_scores[m].mean())
        rows.append({"video_id": vid, "window_count": int(m.sum()), "gate_supported_probability": gate,
                     "oracle_exercise": oracle, "prediction": oracle if gate >= FIXED_THRESHOLD else "unknown"})
    t = pd.DataFrame(rows)
    if truth is not None:
        t["label"] = t.video_id.map(truth)
        t["role_group"] = t.video_id.map(dict(zip(ROLES.video_id, ROLES.role_group)))
        t["error_type"] = [error_type(a, b) for a, b in zip(t.label, t.prediction)]
    return t


def dev_metrics(t):
    recall = {c: float((t.prediction[t.label == c] == c).mean()) for c in TEST_CLASSES if (t.label == c).any()}
    sup = t.label != "unknown"
    return {"clips": int(len(t)),
            "macro_f1_7": float(f1_score(t.label, t.prediction, labels=TEST_CLASSES, average="macro", zero_division=0)),
            "accuracy": float((t.label == t.prediction).mean()),
            "unknown_rejection": recall.get("unknown", float("nan")),
            "gate_false_unknown": int((t.error_type == "gate_false_unknown").sum()),
            "gate_false_accept": int((t.error_type == "gate_false_accept").sum()),
            "wrong_supported_class": int((t.error_type == "wrong_supported_class").sum()),
            "oracle_correct_supported": int((t.oracle_exercise[sup] == t.label[sup]).sum()),
            **{f"recall__{c}": v for c, v in recall.items()}}


## Control: frozen v2, with parity checks


In [ ]:
def build_arm(classifier_state, gate_states):
    cls = SmallTCN(FEATURE_DIM, len(SUPPORTED_CLASSES)).to(DEVICE); cls.load_state_dict(classifier_state); cls.eval()
    gates = {}
    for s in STREAMS:
        g = SmallTCN(STREAMS[s], 2).to(DEVICE); g.load_state_dict(gate_states[s]); gates[s] = g.eval()
    return cls, gates


DEV_POSE = {vid: long_pose(vid)[0] for vid in LONG_DEV}
DEV_ANN = {vid: long_annotation(vid) for vid in LONG_DEV}


def dev_frames(model):
    """Evaluation-1 scorer on each dev video: dense windows, smoothed decoder."""
    out = {}
    for vid in LONG_DEV:
        times, raw, gate_scores = lvscore.predict(DEV_POSE[vid], model, DEVICE)
        truth, seg, guard = lvscore.truth_at(times, DEV_ANN[vid])
        out[vid] = pd.DataFrame({"video_id": vid, "time_s": times, "truth": truth, "segment_type": seg, "guard": guard,
                                 "pred": lvscore.smooth(raw.copy()), "gate": gate_scores})
    return out


with zipfile.ZipFile(PATHS["v2_stage_a.zip"]) as archive:
    v2_internal = pd.read_csv(archive.open("internal_test_metrics.csv"))
    v2_states = {seed: (torch.load(io.BytesIO(archive.read(f"models/seed_{seed}_classifier.pt")), map_location=DEVICE, weights_only=False)["exercise_state_dict"],
                        {s: torch.load(io.BytesIO(archive.read(f"models/seed_{seed}_{s}_gate.pt")), map_location=DEVICE, weights_only=False)["state_dict"] for s in STREAMS})
                 for seed in CONTROL_SEEDS}
diag = pd.read_csv(PATHS["dev_diagnosis_metrics.csv"])
control_frames, control_dev, control_internal = {}, [], []
for seed in CONTROL_SEEDS:
    model = build_arm(*v2_states[seed])
    cls, gates = model
    im = internal_metrics(fused_score(gates, test_idx), class_probabilities(cls, test_idx))
    ref = float(v2_internal.loc[(v2_internal.arm == "treatment") & (v2_internal.seed == seed), "end_to_end_macro_f1"].iloc[0])
    assert abs(im["end_to_end_macro_f1"] - ref) < 0.003, (seed, im["end_to_end_macro_f1"], ref)
    control_frames[seed] = dev_frames(model)
    dm = lvscore.metrics(list(control_frames[seed].values()))
    if not DRY_RUN:
        dref = diag[(diag.model == f"v2_seed{seed}") & (diag.decoder == "smoothed") & (diag.video_id == "ALL")].iloc[0]
        for k in ("macro_f1", "supported_called_unknown", "downtime_called_unknown"):
            assert abs(dm[k] - dref[k]) < 0.005, (seed, k, dm[k], dref[k])
    control_dev.append({"arm": "control_v2", "seed": seed, **dm})
    control_internal.append({"arm": "control_v2", "seed": seed, **im, "nb43_reference_macro_f1": ref})
print("v2 control parity passed (internal test vs Notebook 43; dev vs the dev diagnosis).")
display(pd.DataFrame(control_dev)[["arm", "seed", "macro_f1", "supported_called_unknown", "downtime_called_unknown"]])


## Treatment: frozen Notebook 45 classifiers, retrained balanced gates


In [ ]:
REPORT = WORK / "calorie_long_video_gate_balance_report"
if REPORT.exists():
    shutil.rmtree(REPORT)
(REPORT / "models").mkdir(parents=True)
treatment_frames, treatment_dev, treatment_internal, history_rows, best_rows = {}, [], [], [], []
for seed in RUN_SEEDS:
    with zipfile.ZipFile(PATHS["nb45_report.zip"]) as archive:  # frozen Notebook 45 classifier
        payload = archive.read(f"models/seed_{seed}_classifier.pt")
    nb45_cls = torch.load(io.BytesIO(payload), map_location=DEVICE, weights_only=False)
    assert list(nb45_cls["supported_classes"]) == SUPPORTED_CLASSES and nb45_cls["run_seed"] == seed
    classifier = SmallTCN(FEATURE_DIM, len(SUPPORTED_CLASSES)).to(DEVICE)
    classifier.load_state_dict(nb45_cls["exercise_state_dict"]); classifier.eval()
    (REPORT / "models" / f"seed_{seed}_classifier_from_nb45.pt").write_bytes(payload)
    gates = {}
    for offset, stream in enumerate(STREAMS):
        ts = seed + 100 + offset
        gate, history, best = train_gate(stream, make_stream_loaders(stream, ts, gate_train, gate_weights, validation_indices), ts, "gate")
        gates[stream] = gate
        history_rows += history
        best_rows.append({"seed": seed, "model": stream, "best_validation_macro_f1": best})
        torch.save({"state_dict": gate.state_dict(), "stream": stream, "input_features": STREAMS[stream], "run_seed": seed,
                    "training_seed": ts, "fusion": FIXED_FUSION, "threshold": FIXED_THRESHOLD}, REPORT / "models" / f"seed_{seed}_{stream}_gate.pt")
    treatment_frames[seed] = dev_frames((classifier, gates))
    treatment_dev.append({"arm": "treatment", "seed": seed, **lvscore.metrics(list(treatment_frames[seed].values()))})
    treatment_internal.append({"arm": "treatment", "seed": seed, **internal_metrics(fused_score(gates, test_idx), class_probabilities(classifier, test_idx))})
    del classifier, gates
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()
display(pd.DataFrame(treatment_dev)[["arm", "seed", "macro_f1", "supported_called_unknown", "downtime_called_unknown"]])


## Predeclared dev rule and report


In [ ]:
control_dev_df, treatment_dev_df = pd.DataFrame(control_dev), pd.DataFrame(treatment_dev)
internal = pd.DataFrame(control_internal + treatment_internal)
med = lambda frame, column: float(np.median(frame[column].to_numpy(dtype=float)))

rng = np.random.default_rng(20261027)
deltas = []
for _ in range(2000):
    sample = rng.choice(LONG_DEV, size=len(LONG_DEV), replace=True)
    arm = {name: np.median([lvscore.metrics([fr[v] for v in sample])["macro_f1"] for fr in frames.values()])
           for name, frames in (("control", control_frames), ("treatment", treatment_frames))}
    deltas.append(arm["treatment"] - arm["control"])
ci = (float(np.percentile(deltas, 2.5)), float(np.percentile(deltas, 97.5)))

unknown_drop = med(control_dev_df, "supported_called_unknown") - med(treatment_dev_df, "supported_called_unknown")
f1_gain = med(treatment_dev_df, "macro_f1") - med(control_dev_df, "macro_f1")
downtime_change = med(treatment_dev_df, "downtime_called_unknown") - med(control_dev_df, "downtime_called_unknown")
ti, ci_int = internal[internal.arm == "treatment"], internal[internal.arm == "control_v2"]
internal_change = float(ti.end_to_end_macro_f1.mean() - ci_int.end_to_end_macro_f1.mean())
criteria = {
    "supported_called_unknown_drops_at_least_15pp": unknown_drop >= 0.15,
    "smoothed_macro_f1_gain_at_least_5pp": f1_gain >= 0.05,
    "macro_f1_gain_bootstrap_ci_above_zero": ci[0] > 0,
    "downtime_called_unknown_within_10pp_of_control": downtime_change >= -0.10,
    "internal_macro_f1_within_2pp_of_control": internal_change >= -0.02,
}
with zipfile.ZipFile(PATHS["nb45_report.zip"]) as archive:
    nb45_dev = pd.read_csv(archive.open("dev_metrics_by_seed.csv"))
nb45_dev = nb45_dev[nb45_dev.arm == "treatment"]
vs_nb45 = {k: med(treatment_dev_df, k) - med(nb45_dev, k) for k in ("macro_f1", "supported_called_unknown", "downtime_called_unknown")}
verdict = "dry_run_not_a_result" if DRY_RUN else ("dev_pass" if all(criteria.values()) else "dev_fail")
stopping_rule = {"stage": STAGE, "verdict": verdict, "criteria": {k: bool(v) for k, v in criteria.items()},
                 "supported_called_unknown_drop": unknown_drop, "smoothed_macro_f1_gain": f1_gain,
                 "macro_f1_gain_bootstrap_95ci": ci, "downtime_called_unknown_change": downtime_change,
                 "internal_macro_f1_change": internal_change, "locked_long_video_test_loaded": False,
                 "repcount_sealed_test_loaded": False, "production_promotion_allowed": False,
                 "next_if_pass": "Evaluation 2: score these frozen models once on the locked long-video test.",
                 "next_if_fail": "Report and close this arc (PLAN.md follow-up 1).",
                 "post_hoc_followup": True, "vs_notebook45_reported_only": vs_nb45}
print(json.dumps(stopping_rule, indent=2))
summary = pd.concat([control_dev_df, treatment_dev_df], ignore_index=True)
display(summary[["arm", "seed", "macro_f1", "accuracy", "block_f1", "supported_called_unknown", "downtime_called_unknown"]])
display(internal[["arm", "seed", "end_to_end_macro_f1", "unknown_recall", "uncovered_mean_recall"]])

per_video = []
for name, frames in (("control_v2", control_frames), ("treatment", treatment_frames)):
    for seed, fr in frames.items():
        for v in LONG_DEV:
            per_video.append({"arm": name, "seed": seed, "video_id": v, **lvscore.metrics([fr[v]])})
run_metadata = {
    "notebook": "46_long_video_gate_balance.ipynb", "experiment": "long_video_training_v1_followup1", "long_gate_share": LONG_GATE_SHARE,
    "frozen_classifiers": "Notebook 45 seeds", "nb45_commit": NB45_COMMIT, "stage": STAGE,
    "plan": "ml/data/long_video_train_v1/PLAN.md", "long_commit": LONG_COMMIT, "training_commit": TRAINING_COMMIT,
    "roles_commit": ROLES_COMMIT, "dry_run": DRY_RUN, "device": str(DEVICE), "run_seeds": RUN_SEEDS,
    "control": "frozen v2 seeds (Notebook 43)", "max_epochs": MAX_EPOCHS, "patience": PATIENCE,
    "fixed_fusion": FIXED_FUSION, "fixed_threshold": FIXED_THRESHOLD, "threshold_changed": False, "repcount_cap": REPCOUNT_CAP,
    "long_label_share": LONG_LABEL_SHARE, "long_detected_share": LONG_DETECTED_SHARE,
    "long_train_videos": LONG_TRAIN, "long_dev_videos": LONG_DEV, "long_train_windows": int(len(long_train_idx)),
    "locked_benchmark_read": False, "locked_long_video_test_read": False, "repcount_sealed_test_loaded": False,
    "dev_used_for_early_stopping": False, "internal_test_used_for_selection": False, "production_promoted": False,
    "input_hashes": {name: sha256(path) for name, path in PATHS.items()},
}
(REPORT / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2) + "\n")
(REPORT / "stopping_rule.json").write_text(json.dumps(stopping_rule, indent=2) + "\n")
summary.to_csv(REPORT / "dev_metrics_by_seed.csv", index=False)
pd.DataFrame(per_video).to_csv(REPORT / "dev_metrics_by_video.csv", index=False)
for name, frames in (("control_v2", control_frames), ("treatment", treatment_frames)):
    for seed, fr in frames.items():
        pd.concat(fr.values()).to_csv(REPORT / f"dev_timeline_{name}_seed{seed}_smoothed.csv.gz", index=False)
internal.to_csv(REPORT / "internal_test_metrics.csv", index=False)
long_audit.to_csv(REPORT / "long_video_window_audit.csv", index=False)
sampler_audit.to_csv(REPORT / "sampler_audit.csv", index=False)
pd.DataFrame(history_rows).to_csv(REPORT / "training_history.csv", index=False)
pd.DataFrame(best_rows).to_csv(REPORT / "best_validation.csv", index=False)
pd.DataFrame({"delta": deltas}).to_csv(REPORT / "bootstrap_macro_f1_delta.csv", index=False)
(REPORT / "README.md").write_text(
    "# cALorie long-video gate balance, follow-up 1 (Notebook 46)\n\n"
    + ("DRY RUN: one epoch on subsamples. Not a result.\n\n" if DRY_RUN else "")
    + "Notebook 45 classifiers frozen; gates retrained with long-video windows at 25% of each gate class; control is frozen v2. Scored on the 5 dev videos "
    "(full-session scorer, smoothed) and the internal test. No locked or sealed test was read. Non-promotable. Verdict: " + verdict + "\n"
)
lines = [f"{sha256(p)}  {p.relative_to(REPORT)}" for p in sorted(q for q in REPORT.rglob("*") if q.is_file())]
(REPORT / "SHA256SUMS.txt").write_text("\n".join(lines) + "\n")
zip_path = Path(shutil.make_archive(str(WORK / ("calorie_long_video_gate_balance_report" + ("_DRYRUN" if DRY_RUN else ""))), "zip", root_dir=REPORT))
print("Created:", zip_path, round(zip_path.stat().st_size / 1e6, 1), "MB")
